# 🌊 SIPI AI — Análisis de Predicción de Inundaciones en Chile

**Equipo 7**: Carolina Pino · Felipe Paimilla · Débora Cáceres · Bastián Figueroa

---

## Pregunta de Análisis

> *¿Es posible predecir crecidas fluviales en ríos de Chile con 24 a 48 horas de anticipación utilizando exclusivamente datos meteorológicos satelitales (ERA5) y registros hidrométricos históricos (CAMELS-CL) como insumos de modelos XGBoost?*

## Datasets

| Dataset | Fuente | Descripción |
| :--- | :--- | :--- |
| **CAMELS-CL** | CR2 / DGA | Caudales diarios (m³/s) de 516 estaciones + atributos de cuenca |
| **ERA5 (Open-Meteo)** | ECMWF | Reanálisis climático: precipitación, temperatura, humedad del suelo, nieve |
| **Open-Meteo Forecast** | Open-Meteo | Pronóstico meteorológico a 3 días para predicción en vivo |

## 1. Importación de Librerías

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import json
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

print(f"XGBoost version: {xgb.__version__}")
print(f"Pandas version: {pd.__version__}")

## 2. Carga y Exploración de Datos

Los datasets de entrenamiento combinan datos de caudal (CAMELS-CL) con variables climáticas satelitales (ERA5). Cada CSV contiene una estación fluviométrica con su historial diario.

In [ ]:
# Listar todos los datasets de entrenamiento disponibles
DATOS_DIR = "datos_entrenamiento"
archivos_csv = sorted(glob.glob(os.path.join(DATOS_DIR, "datos_entrenamiento_*.csv")))

print(f"Total de datasets de entrenamiento encontrados: {len(archivos_csv)}")
print(f"\nPrimeros 5 archivos:")
for f in archivos_csv[:5]:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  📄 {os.path.basename(f)} ({size_mb:.2f} MB)")

In [ ]:
# Cargar un dataset de ejemplo para explorar su estructura
df_ejemplo = pd.read_csv(archivos_csv[0])

print(f"Dimensiones: {df_ejemplo.shape[0]} filas × {df_ejemplo.shape[1]} columnas")
print(f"\nColumnas del dataset:")
for i, col in enumerate(df_ejemplo.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\nPrimeras filas:")
df_ejemplo.head()

In [ ]:
# Estadísticas descriptivas de las variables principales
cols_interes = [col for col in ['Caudal_CAMELS_m3_s', 'precipitacion_era5_mm', 'temperatura_c', 
                                'temperatura_zona_alta_c', 'swe_mm', 'humedad_suelo_capa1_vol'] 
                if col in df_ejemplo.columns]

df_ejemplo[cols_interes].describe().round(3)

## 3. Feature Engineering

El proceso de ingeniería de características transforma los datos crudos en variables que capturan la dinámica hidrológica:

- **Lags temporales**: Caudal y clima de los últimos 1-3 días (memoria hidrológica)
- **Acumuladores**: Precipitación acumulada a 3 y 7 días (saturación del suelo)
- **Sensor virtual de Isoterma Cero**: Detecta lluvia cálida sobre nieve en la cordillera
- **Variables de pronóstico**: Valores futuros desplazados (shift) para horizonte 24h/48h

In [ ]:
def aplicar_feature_engineering(df, shift=-1):
    """
    Genera variables derivadas para capturar la dinámica hidrológica.
    - shift=-1 para horizonte 24h, shift=-2 para horizonte 48h
    """
    df = df.copy()
    
    # Variables de pronóstico (desplazar el futuro hacia atrás)
    cols_clima = ['precipitacion_era5_mm', 'temperatura_c']
    if 'precipitacion_zona_alta_mm' in df.columns:
        cols_clima.extend(['precipitacion_zona_alta_mm', 'temperatura_zona_alta_c'])
    
    for col in cols_clima:
        if col in df.columns:
            df[f'{col}_pronostico'] = df[col].shift(shift)
    
    # Sensor virtual de Isoterma Cero Alta
    if 'temperatura_zona_alta_c_pronostico' in df.columns and 'precipitacion_zona_alta_mm_pronostico' in df.columns:
        df['riesgo_isoterma0_alta_pronostico'] = (
            (df['temperatura_zona_alta_c_pronostico'] > 0.0) & 
            (df['precipitacion_zona_alta_mm_pronostico'] > 2.0)
        ).astype(int)
    
    # Lags de 1 a 3 días
    excl = {'Fecha', 'Caudal_CAMELS_m3_s', 'caudal_futuro', 'inundacion_futura', 'caudal_hoy'}
    cols_lag = [c for c in df.columns if c not in excl and not c.endswith('_pronostico')]
    for lag in range(1, 4):
        for col in cols_lag:
            df[f'{col}_lag{lag}'] = df[col].shift(lag)
    
    # Lags de caudal
    if 'Caudal_CAMELS_m3_s' in df.columns:
        for lag in range(1, 4):
            df[f'caudal_lag{lag}'] = df['Caudal_CAMELS_m3_s'].shift(lag)
    
    # Acumuladores de precipitación
    if 'precipitacion_era5_mm' in df.columns:
        df['precipitacion_acum_3d'] = df['precipitacion_era5_mm'].rolling(3).sum()
        df['precipitacion_acum_7d'] = df['precipitacion_era5_mm'].rolling(7).sum()
    
    return df

# Aplicar Feature Engineering al dataset de ejemplo
df_fe = aplicar_feature_engineering(df_ejemplo.copy(), shift=-1)
df_fe = df_fe.dropna()

print(f"Columnas originales: {df_ejemplo.shape[1]}")
print(f"Columnas después de Feature Engineering: {df_fe.shape[1]}")
print(f"Features nuevas generadas: {df_fe.shape[1] - df_ejemplo.shape[1]}")
print(f"Filas válidas (sin NaN por lags/shifts): {len(df_fe)}")

## 4. Entrenamiento del Modelo XGBoost

Se entrenan **dos modelos complementarios** por cada estación:

1. **XGBRegressor**: Predice el caudal exacto en m³/s
2. **XGBClassifier**: Estima la probabilidad de crecida (superar el Percentil 95 histórico)

La división temporal (80/20) se realiza **sin shuffle** para respetar la secuencia cronológica y evitar data leakage.

In [ ]:
# Preparar features y target
features = [col for col in df_fe.columns if col not in ['Fecha', 'Caudal_CAMELS_m3_s']]
X = df_fe[features]
y_caudal = df_fe['Caudal_CAMELS_m3_s']

# Variable binaria de alerta (Percentil 95)
umbral_p95 = y_caudal.quantile(0.95)
y_alerta = (y_caudal >= umbral_p95).astype(int)

print(f"Total de features: {len(features)}")
print(f"Umbral P95 de caudal: {umbral_p95:.2f} m³/s")
print(f"Eventos de crecida (>=P95): {y_alerta.sum()} ({y_alerta.mean()*100:.1f}%)")

# División temporal 80/20 (sin shuffle)
X_train, X_test, y_caudal_train, y_caudal_test = train_test_split(
    X, y_caudal, test_size=0.2, shuffle=False
)
_, _, y_alerta_train, y_alerta_test = train_test_split(
    X, y_alerta, test_size=0.2, shuffle=False
)

print(f"\nConjunto de entrenamiento: {len(X_train)} muestras")
print(f"Conjunto de prueba: {len(X_test)} muestras")

In [ ]:
# Entrenar Regresor XGBoost (predicción de caudal en m³/s)
modelo_regresor = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    tree_method='hist',
    random_state=42
)
modelo_regresor.fit(X_train, y_caudal_train)

# Entrenar Clasificador XGBoost (probabilidad de crecida)
modelo_clasificador = xgb.XGBClassifier(
    n_estimators=150,
    learning_rate=0.01,
    max_depth=5,
    scale_pos_weight=15,  # Balanceo de clases (los desbordes son eventos raros)
    tree_method='hist',
    random_state=42
)
modelo_clasificador.fit(X_train, y_alerta_train)

print("✅ Regresor XGBoost entrenado exitosamente")
print("✅ Clasificador XGBoost entrenado exitosamente")

## 5. Evaluación del Modelo (Datos Out-of-Sample)

In [ ]:
# Predicciones sobre el conjunto de prueba
y_pred_caudal = modelo_regresor.predict(X_test)
y_pred_prob = modelo_clasificador.predict_proba(X_test)[:, 1]

# Métricas del Regresor
r2 = r2_score(y_caudal_test, y_pred_caudal)
rmse = np.sqrt(mean_squared_error(y_caudal_test, y_pred_caudal))
mae = mean_absolute_error(y_caudal_test, y_pred_caudal)

# Nash-Sutcliffe Efficiency (NSE)
nse_num = np.sum((y_caudal_test.values - y_pred_caudal) ** 2)
nse_den = np.sum((y_caudal_test.values - np.mean(y_caudal_test.values)) ** 2)
nse = 1 - (nse_num / nse_den) if nse_den != 0 else 0

# Métrica del Clasificador
try:
    auc = roc_auc_score(y_alerta_test, y_pred_prob)
except:
    auc = float('nan')

print("=" * 55)
print("  MÉTRICAS DE EVALUACIÓN (Datos No Vistos - Test Set)")
print("=" * 55)
print(f"  NSE (Nash-Sutcliffe):   {nse:.4f}")
print(f"  R² (Determinación):     {r2:.4f}")
print(f"  RMSE (m³/s):            {rmse:.4f}")
print(f"  MAE (m³/s):             {mae:.4f}")
print(f"  ROC-AUC (Clasificador): {auc:.4f}")
print("=" * 55)

In [ ]:
# Visualización: Caudal Observado vs Predicho
fig = go.Figure()

fig.add_trace(go.Scatter(
    y=y_caudal_test.values, name='Observado',
    line=dict(color='#34d399', width=1.5),
    opacity=0.8
))

fig.add_trace(go.Scatter(
    y=y_pred_caudal, name='Predicho XGBoost',
    line=dict(color='#4da6ff', width=1.5, dash='dash'),
    opacity=0.8
))

fig.add_hline(y=umbral_p95, line_dash='dot', line_color='#fb923c',
              annotation_text=f'Umbral P95 ({umbral_p95:.1f} m³/s)')

fig.update_layout(
    title=f'Caudal Observado vs Predicho — Test Set (NSE={nse:.3f}, R²={r2:.3f})',
    xaxis_title='Día (secuencial)',
    yaxis_title='Caudal (m³/s)',
    template='plotly_dark',
    height=450,
    legend=dict(x=0.01, y=0.99)
)

fig.show()

## 6. Importancia de Variables (Feature Importance)

XGBoost permite inspeccionar qué variables fueron más relevantes para las predicciones. Esto es clave para la **explicabilidad** del modelo.

In [ ]:
# Feature Importance del Regresor
importancias = modelo_regresor.feature_importances_
df_imp = pd.DataFrame({
    'Feature': features,
    'Importancia': importancias
}).sort_values('Importancia', ascending=True).tail(15)

fig_imp = px.bar(
    df_imp, x='Importancia', y='Feature', orientation='h',
    title='Top 15 Variables Más Importantes (XGBoost Regresor)',
    color='Importancia',
    color_continuous_scale='viridis',
    template='plotly_dark'
)
fig_imp.update_layout(height=500, showlegend=False, yaxis_title='')
fig_imp.show()

## 7. Resultados Nacionales

El pipeline completo entrenó modelos para **392 estaciones** a nivel nacional. A continuación se presentan los resultados agregados del informe de evaluación.

In [ ]:
# Resultados nacionales del informe de evaluación
resultados_nacionales = {
    'Métrica': ['NSE (Nash-Sutcliffe)', 'R² (Determinación)', 'ROC-AUC (Crecidas P95)'],
    'Mediana Nacional': [0.810, 0.810, 0.917],
    'Categoría': ['🟢 Excelente', '🟢 Excelente', '🟢 Alta Precisión (>90%)']
}

df_resumen = pd.DataFrame(resultados_nacionales)
print("\n📊 RESUMEN DE RENDIMIENTO NACIONAL")
print("=" * 60)
display(df_resumen)

# Distribución de calidad
distribucion = {
    'Categoría': ['🟢 Excelente (NSE > 0.75)', '🔵 Bueno (0.5-0.75)', 
                  '🟡 Moderado (0-0.5)', '🔴 Deficiente (NSE ≤ 0)'],
    'Cantidad': [243, 70, 49, 46],
    'Porcentaje': [59.6, 17.2, 12.0, 11.3]
}

df_dist = pd.DataFrame(distribucion)
print("\n📊 DISTRIBUCIÓN DE CALIDAD DE MODELOS")
print("=" * 60)
display(df_dist)

In [ ]:
# Visualización de la distribución de calidad
colores = ['#34d399', '#60a5fa', '#fbbf24', '#f87171']

fig_dist = px.pie(
    df_dist, values='Cantidad', names='Categoría',
    title='Distribución de Calidad de Modelos a Nivel Nacional (392 estaciones)',
    color_discrete_sequence=colores,
    template='plotly_dark',
    hole=0.4
)
fig_dist.update_traces(textposition='outside', textinfo='percent+label')
fig_dist.update_layout(height=500, showlegend=False)
fig_dist.show()

## 8. Conclusiones

### Respuesta a la Pregunta de Análisis

**Sí, es posible predecir crecidas fluviales en ríos de Chile con 24 a 48 horas de anticipación** utilizando datos meteorológicos satelitales y modelos XGBoost.

Los resultados nacionales lo confirman:

- **76.8%** de las estaciones (313 de 392) obtuvieron un rendimiento Bueno o Excelente (NSE > 0.5)
- La **mediana del NSE nacional es 0.810**, lo cual es considerado "Excelente" en hidrología según la clasificación de Moriasi et al. (2007)
- El **clasificador de crecidas alcanzó un AUC de 0.917**, lo que significa que discrimina correctamente entre eventos de crecida y normalidad en el 91.7% de los casos

### Limitaciones

- **Disponibilidad de Datos ERA5**: Por limitaciones de tiempo en el desarrollo, no fue posible procesar y descargar la serie temporal climática completa para la totalidad de las cuencas. En aquellas estaciones donde solo se utilizó el registro hidrométrico histórico de CAMELS sin el enriquecimiento satelital, la efectividad predictiva del modelo se vio significativamente reducida.
- Las estaciones con rendimiento deficiente (11.3%) corresponden principalmente a:
  - Ríos hiperáridos del norte de Chile (Atacama, Antofagasta) con caudales cercanos a cero
  - Estaciones con menos de 200 registros válidos
  - Cauces de régimen nival extremo donde el deshielo glaciar introduce alta variabilidad
- XGBoost tiene una limitación teórica: **no puede extrapolar** más allá del rango de valores observados en el entrenamiento, lo que podría subestimar eventos de crecida sin precedentes históricos

### Trabajo Futuro

- Incorporar datos de radar meteorológico para mejorar la predicción en zonas de alta montaña
- Explorar modelos LSTM o Transformers que capturen mejor las dependencias temporales de largo plazo
- Integrar modelos hidrológicos físicos (GR4J) como baseline de comparación